In [1]:
from src.core.market_calendar import USMarketsCalendar
from src.core.market_clock import MarketClock, AcceleratedTimeHeartbeat
from src.core.system_manager import SystemManager

from src.data.data_access import InMemorySessionDAL
from src.data.data_ingestion import InMemorySessionDIL

from src.monitoring.health_service import build_readiness_checks


checks = build_readiness_checks(db_connector=None, ibkr_gateway=None, eodhd_client=None)


# heartbeat = SystemTimeHeartbeat()
sim_heartbeat = AcceleratedTimeHeartbeat()

market_clock = MarketClock(time_source=sim_heartbeat)
market_calendar = USMarketsCalendar()

data_ingestion = InMemorySessionDIL()
data_access = InMemorySessionDAL()


system_manager = SystemManager(market_clock=market_clock,
                               market_calendar=market_calendar,
                               data_ingestion=data_ingestion,
                               data_access=data_access)


# system_manager._market_clock.start()

system_manager._prepare_bootstrap(checks=checks)

system_manager.launch_global_runtime()

system_manager.launch_local_runtime()


In [2]:
system_manager._modules.ibkr_gateway.ports.data_access.get_universe_snapshot()

UniverseSnapshot(universe_id='SP500', as_of=datetime.datetime(2026, 1, 18, 23, 37, 44, 62715, tzinfo=datetime.timezone.utc), constituents={'EQ_US_AAPL': TickerUniverseSnapshot(internal_code='EQ_US_AAPL', symbol='AAPL', asset_type=<AssetType.STOCK: 'STOCK'>, tickers=['AAPL'], updated_at=datetime.datetime(2026, 1, 18, 23, 37, 44, 62715, tzinfo=datetime.timezone.utc)), 'EQ_US_MSFT': TickerUniverseSnapshot(internal_code='EQ_US_MSFT', symbol='MSFT', asset_type=<AssetType.STOCK: 'STOCK'>, tickers=['MSFT'], updated_at=datetime.datetime(2026, 1, 18, 23, 37, 44, 62715, tzinfo=datetime.timezone.utc)), 'EQ_US_NVDA': TickerUniverseSnapshot(internal_code='EQ_US_NVDA', symbol='NVDA', asset_type=<AssetType.STOCK: 'STOCK'>, tickers=['NVDA'], updated_at=datetime.datetime(2026, 1, 18, 23, 37, 44, 62715, tzinfo=datetime.timezone.utc))})

In [7]:
from src.models.market_data import MarketTick, MarketDataInstrument
from src.trading.ibkr_gateway import MockMarketDataSubscriber
import time

def on_tick(tick: MarketTick) -> None:
    print(tick)

sub = MockMarketDataSubscriber(interval_s=0.3)
sub.start()

handle = sub.subscribe(
    MarketDataInstrument(internal_code="AAPL", symbol="AAPL", asset_type="EQ"),
    on_tick,
)

time.sleep(10.0)
sub.unsubscribe(handle)
sub.stop()


MarketTick(symbol='AAPL', ts=datetime.datetime(2026, 1, 19, 10, 5, 35, 853967, tzinfo=datetime.timezone.utc), last=100.46389892581685, bid=100.45889892581685, ask=100.46889892581684)
MarketTick(symbol='AAPL', ts=datetime.datetime(2026, 1, 19, 10, 5, 36, 157053, tzinfo=datetime.timezone.utc), last=100.3665887615911, bid=100.36099335787308, ask=100.37218416530912)
MarketTick(symbol='AAPL', ts=datetime.datetime(2026, 1, 19, 10, 5, 36, 459679, tzinfo=datetime.timezone.utc), last=100.34770209906445, bid=100.34270209906445, ask=100.35270209906444)
MarketTick(symbol='AAPL', ts=datetime.datetime(2026, 1, 19, 10, 5, 36, 764095, tzinfo=datetime.timezone.utc), last=100.29380554730284, bid=100.28849129156772, ask=100.29911980303795)
MarketTick(symbol='AAPL', ts=datetime.datetime(2026, 1, 19, 10, 5, 37, 69321, tzinfo=datetime.timezone.utc), last=100.3764770841032, bid=100.3714770841032, ask=100.3814770841032)
MarketTick(symbol='AAPL', ts=datetime.datetime(2026, 1, 19, 10, 5, 37, 373274, tzinfo=date